In [ ]:
import sys
# sys.path.insert(0, '../../Utilities/')

# from jax.example_libraries import optimizers
import jax.numpy as jnp
import numpy as np
from jax import grad, jit, vmap, value_and_grad, hessian
from jax import random
from functools import partial
import flax.linen as nn
import jax
import optax
import pickle
from datetime import datetime
import os
import numpy as np
from scipy import sparse
import time
from jax import grad, hessian, jit, vmap, pmap
from jax.tree_util import tree_leaves
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import flax.linen as nn
import optax
import optax.tree_utils as otu
from NN_class import MLP
from my_helpers import plot_u_theta, plot_kh, plot_loss_history
from functools import partial
from Pwp_class import PiecewiseLinearPolynomials
from minmax_opt_class import pytree_l2_norm, MinMaxOptProb
from seq_minmax_opt_class import SequentialMinMaxOptProb


import matplotlib.pyplot as plt
import scipy.io
from scipy.interpolate import griddata

import time
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import mpmath

# Generate key which is used to generate random numbers
key = random.PRNGKey(1)

In [ ]:
# def u0(x):
#     u = jnp.zeros_like(x)
#     u = jnp.where((x > 0.0) & (x < 1.0), 1.0, u)
#     return u

def u0(x):
    # return 2*smooth_sign(x) - 1
    return -jnp.sign(x)

def u_exact(t, x):
    # Initialize u as zero
    u = 0.0
    # Condition 1: x < 0, u = 0 (already zero by default)
    # Condition 2: 0 < x < t and t > 0, u = x/t
    u = jnp.where((x > 0) & (x < t) & (t > 0), x / t, u)
    
    # Condition 3: t < x < 1 + t/2, u = 1
    u = jnp.where((x > t) & (x < 1 + t / 2), 1.0, u)
    
    # Condition 4: x > 1 + t/2, u = 0 (already zero by default)
    return u
vmap_u_exact = vmap(u_exact, in_axes=(0, 0), out_axes=0)

# Boundary at x=-1
def u1(x):
    return 1
vmap_u1 = vmap(u1, in_axes=0, out_axes=0)


# Boundary at x=1
def u2(x):
    return -1
vmap_u2 = vmap(u2, in_axes=0, out_axes=0)

def res_visc(u_quad, u_t, u_x, u_xx, nu):
    return u_t + u_quad * u_x - nu*u_xx
def res(u_quad, u_t, u_x):
    return u_t + u_quad * u_x


t_min = 0.0
t_max = 0.5
# num_segments = 6
x_min = -1
x_max = 1
Nt = 101
Nx = 401
layers = [2, 50, 50, 50, 50, 50, 1]
hidden_dims = layers[1:-1]

# Domain bounds
lb = jnp.array([t_min, x_min])
ub = jnp.array([t_max, x_max])

In [ ]:
optProb = MinMaxOptProb(
    N_t=16,
    N_x=64,
    lr_theta=0.002,
    seed=0,
    N_bc=100,
    N_ic=400,
    t_min=0, t_max=0.5,  x_min=x_min, x_max=x_max,
    hidden_dims=hidden_dims,
    u0=u0, u1=vmap_u1, u2=vmap_u2,
    nu=0.005,
    res_visc=res_visc,
    res=res,
    use_loss_elem_int=False
)


In [ ]:
optProb.pwp.coeffs

In [ ]:
seq_trainer = SequentialMinMaxOptProb(
    num_segments=num_segments,
    total_t_min=t_min,
    total_t_max=t_max,
    seq_T = [0,1,2,2.5,3.0],
    N_t=16,
    N_x=48,
    lr_theta=0.0002,
    hidden_dims=hidden_dims,
    seed=0,
    N_bc=Nt,
    N_ic=Nx,
    x_min=x_min, x_max=x_max,
    u0=u0, u1=vmap_u1, u2=vmap_u2,
    nu=0.005,
    res_visc=res_visc,
    res=res,
)
seq_trainer.train_sequential(train_num_iterations=0, pre_train_num_iterations=10000)

In [ ]:
seq_trainer_us = SequentialMinMaxOptProb(
    num_segments=num_segments,
    total_t_min=t_min,
    total_t_max=t_max,
    seq_T = [0, 0.5, 1, 1.5, 2, 2.5, 3.0],
    N_t=16,
    N_x=48,
    lr_theta=0.0002,
    hidden_dims=hidden_dims,
    seed=0,
    N_bc=Nt,
    N_ic=Nx,
    x_min=x_min, x_max=x_max,
    u0=u0, u1=vmap_u1, u2=vmap_u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,
)

seq_trainer_us.train_sequential(train_num_iterations=5000, pre_train_num_iterations=0, use_loss_elem_int=True)

In [ ]:
seq_trainer_us = SequentialMinMaxOptProb(
    num_segments=num_segments,
    total_t_min=t_min,
    total_t_max=t_max,
    seq_T = [0, 0.5, 1, 1.5, 2, 2.5, 3.0],
    N_t=16,
    N_x=48,
    lr_theta=0.0002,
    hidden_dims=hidden_dims,
    seed=0,
    N_bc=Nt,
    N_ic=Nx,
    x_min=x_min, x_max=x_max,
    u0=u0, u1=vmap_u1, u2=vmap_u2,
    nu=0.01,
    res_visc=res_visc,
    res=res, 
    use_loss_elem_int=True,
)

seq_trainer_us.train_sequential(train_num_iterations=5000, pre_train_num_iterations=0)

In [ ]:
seq_trainer_us.plot_solution_at_time(seq_trainer_us.total_t_max)